# Construir el índice en Colab (GPU)

Calcula los embeddings `BAAI/bge-m3` de los fragmentos del corpus en una GPU gratuita
de Colab y devuelve el índice listo para copiar al proyecto. El enunciado (§3.3) permite
usar la capa gratuita de Colab durante la semana.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución →
GPU T4**, y guardar. Luego ejecutar las celdas en orden (Shift + Enter).

En el computador, antes de abrir este cuaderno:
```
python -m src.corpus.build
python -m src.index.paquete_colab      # crea dist/paquete_colab.zip
```

In [ ]:
# 1. Comprobar que hay GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Subir `dist/paquete_colab.zip` (se abre un selector de archivos)

In [ ]:
from google.colab import files
subido = files.upload()
assert "paquete_colab.zip" in subido, "Suba el archivo dist/paquete_colab.zip"
!rm -rf /content/proyecto && mkdir -p /content/proyecto
!unzip -q paquete_colab.zip -d /content/proyecto
%cd /content/proyecto
!ls build/indice

## 3. Instalar dependencias

Se usa el `torch` con CUDA que ya trae Colab (no se reinstala) y las mismas versiones del
proyecto para lo demás, de modo que los embeddings sean equivalentes.

In [ ]:
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0

## 4. Construir el índice en la GPU

In [ ]:
import os
os.environ["ENCODER_DEVICE"] = "cuda"
os.environ["ENCODER_BATCH_SIZE"] = "64"
!python -m src.index.build

## 5. Descargar el resultado

Descarga `indice_colab.zip` con el índice FAISS, el BM25, el manifiesto y la caché de
embeddings. En el computador, descomprimirlo **en la raíz del proyecto** (crea o reemplaza
`build/indice/` y `build/cache/emb/`).

In [ ]:
!cd /content/proyecto && zip -q -r /content/indice_colab.zip build/indice/index.faiss build/indice/index_manifest.json build/indice/bm25 build/cache/emb
!ls -la /content/indice_colab.zip
from google.colab import files
files.download("/content/indice_colab.zip")